# Data Retention and Deletion Workflow - Demo

**Scenario**: PharmaSafe AI — Pharmaceutical company managing clinical trial data

This demo shows how to process a GDPR right-to-be-forgotten request, track model-data lineage, and generate an audit trail.

---

### Quick Reference: Key Concepts

**GDPR Right to Erasure (Art. 17)**: Data subjects can request deletion of their personal data when:
- Data is no longer necessary for its original purpose
- Consent is withdrawn
- Data was unlawfully processed

**Exceptions to Deletion**: Data may be retained when required for:
- Legal obligations (e.g., FDA safety data retention)
- Public health purposes
- Archiving in the public interest
- Defense of legal claims

**Data Retention Principles**:
- Storage Limitation (Art. 5(1)(e)): Keep only as long as necessary
- Purpose Limitation: Data collected for specified purposes only
- Data Minimization: Adequate, relevant, and limited to what is necessary

**Model-Data Lineage**: Tracks which training data was used by which models, enabling impact assessment when data must be deleted.

## Import Libraries

In [ ]:
import pandas as pd
from datetime import datetime, timedelta
import uuid
import matplotlib.pyplot as plt

# Display options
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

## Sample Data Inventory

In [ ]:
# Load data from Excel file (L6-R003: Load data from Excel in demo notebook)
data_retention_policy = pd.read_excel('data_retention_demo.xlsx', sheet_name='Data Retention Policy')
retraining_triggers = pd.read_excel('data_retention_demo.xlsx', sheet_name='Retraining Triggers')

print("Data Retention Policy loaded from Excel:")
print(data_retention_policy[['Data Type', 'Retention Period', 'Deletion Method']].head().to_string(index=False))

# Create PharmaSafe AI data inventory with data_usage field (L6-R002: Add data_usage field)
data_inventory = pd.DataFrame([
    {
        'data_id': 'PART-001',
        'subject_id': 'SUBJ-5001',
        'data_type': 'Participant Demographics',
        'storage_location': 'clinical_db.participants',
        'data_size_mb': 0.5,
        'is_sensitive': True,
        'affected_models': ['Adverse Event Detector v3.2'],
        'data_usage': 'pre_training'  # Used in initial model training
    },
    {
        'data_id': 'TRIAL-001',
        'subject_id': 'SUBJ-5001',
        'data_type': 'Clinical Trial Results',
        'storage_location': 'trial_db.results',
        'data_size_mb': 2.3,
        'is_sensitive': True,
        'affected_models': ['Adverse Event Detector v3.2', 'Signal Detection v1.5'],
        'data_usage': 'pre_training'
    },
    {
        'data_id': 'TRAIN-001',
        'subject_id': 'SUBJ-5001',
        'data_type': 'Model Training Dataset',
        'storage_location': 's3://pharma_ml/training',
        'data_size_mb': 15.8,
        'is_sensitive': True,
        'affected_models': ['Adverse Event Detector v3.2'],
        'data_usage': 'fine_tuning'  # Used for model fine-tuning
    },
    {
        'data_id': 'AUDIT-001',
        'subject_id': 'SUBJ-5001',
        'data_type': 'Audit Logs',
        'storage_location': 'log_db.audit_trail',
        'data_size_mb': 0.2,
        'is_sensitive': False,
        'affected_models': [],
        'data_usage': 'inference_log'  # Contains model inference logs
    }
])

print("\nPharmaSafe AI Data Inventory:")
print(data_inventory.to_string(index=False))

## Process Deletion Request

In [ ]:
# Process deletion request for SUBJ-5001
request_id = str(uuid.uuid4())[:8]
subject_id = 'SUBJ-5001'
request_date = datetime.now()

# Filter data for subject
subject_data = data_inventory[data_inventory['subject_id'] == subject_id]

print(f"Deletion Request ID: {request_id}")
print(f"Subject ID: {subject_id}")
print(f"Request Date: {request_date.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"\nRecords Found: {len(subject_data)}")
print(f"Total Data Size: {subject_data['data_size_mb'].sum():.1f} MB")
print(f"\nAffected Records:")
for idx, row in subject_data.iterrows():
    print(f"  - {row['data_id']}: {row['data_type']} ({row['data_size_mb']} MB)")

## Check Model-Data Lineage

# Assess model impact with data_usage context (L6-R002: Factor in usage type)
affected_models_set = set()
affected_data_usage = {}
for idx, row in subject_data.iterrows():
    for model in row['affected_models']:
        affected_models_set.add(model)
        if model not in affected_data_usage:
            affected_data_usage[model] = []
        affected_data_usage[model].append({
            'data_id': row['data_id'],
            'data_usage': row['data_usage']
        })

affected_models_list = list(affected_models_set)

# Define model production status
model_status = {
    'Adverse Event Detector v3.2': 'in_production',
    'Signal Detection v1.5': 'in_production'
}

print("Model-Data Lineage Analysis with Data Usage Context:")
print(f"\nAffected Models: {len(affected_models_list)}")

models_needing_retraining = []
for model in affected_models_list:
    status = model_status.get(model, 'unknown')
    needs_retraining = status == 'in_production'
    
    # Assess deletion impact based on data_usage (L6-R002)
    data_impacts = affected_data_usage.get(model, [])
    usage_types = [d['data_usage'] for d in data_impacts]
    
    print(f"  - {model} [{status}]" + (" [RETRAINING REQUIRED]" if needs_retraining else ""))
    print(f"    Data usage types involved: {', '.join(set(usage_types))}")
    
    # Deletion impact assessment based on data_usage
    if 'pre_training' in usage_types or 'fine_tuning' in usage_types:
        print(f"    Impact level: HIGH — Must retrain to remove learned patterns")
    elif 'rag_corpus' in usage_types:
        print(f"    Impact level: MEDIUM — RAG index must be rebuilt")
    elif 'prompt_history' in usage_types:
        print(f"    Impact level: MEDIUM — Check for PII leakage in logs")
    else:
        print(f"    Impact level: LOW — Inference log deletion is straightforward")
    
    if needs_retraining:
        models_needing_retraining.append(model)

print(f"\nModels Needing Retraining: {len(models_needing_retraining)}")
for model in models_needing_retraining:
    print(f"  - {model}")

# Show lineage mapping with data usage
print("\nLineage Mapping (with data usage):")
for idx, row in subject_data.iterrows():
    if row['affected_models']:
        print(f"  {row['data_id']} [{row['data_usage']}] -> {', '.join(row['affected_models'])}")
    else:
        print(f"  {row['data_id']} [{row['data_usage']}] -> [No affected models]")

In [ ]:
# Identify affected models from the data
affected_models_set = set()
for models_list in subject_data['affected_models']:
    affected_models_set.update(models_list)

affected_models_list = list(affected_models_set)

# Define model production status
model_status = {
    'Adverse Event Detector v3.2': 'in_production',
    'Signal Detection v1.5': 'in_production'
}

print("Model-Data Lineage Analysis:")
print(f"\nAffected Models: {len(affected_models_list)}")

models_needing_retraining = []
for model in affected_models_list:
    status = model_status.get(model, 'unknown')
    needs_retraining = status == 'in_production'
    if needs_retraining:
        models_needing_retraining.append(model)
    print(f"  - {model} [{status}]" + (" [RETRAINING REQUIRED]" if needs_retraining else ""))

print(f"\nModels Needing Retraining: {len(models_needing_retraining)}")
for model in models_needing_retraining:
    print(f"  - {model}")

# Show lineage mapping
print("\nLineage Mapping:")
for idx, row in subject_data.iterrows():
    if row['affected_models']:
        print(f"  {row['data_id']} -> {', '.join(row['affected_models'])}")
    else:
        print(f"  {row['data_id']} -> [No affected models]")

## The Unlearning Problem: Data Deletion & Model Weights (L6-R002 + L6-R009: GenAI Nativeness)

When GenAI models are trained on personal data, deleting that data creates a unique governance challenge called the **unlearning problem**. This notebook now tracks `data_usage` to assess the impact of deletion on different model components.

### Data Usage Types and Deletion Implications

1. **pre_training**: Data used in the base model's initial training phase
   - Deletion impact: Cannot be undone; learned patterns remain in model weights
   - Governance: High risk for privacy breaches; deletion doesn't guarantee "forgetting"
   - Action: Must retrain from scratch with filtered dataset

2. **fine_tuning**: Data used to specialize the model for a specific task
   - Deletion impact: Fine-tuned patterns remain; deletion requires model reset + re-fine-tuning
   - Governance: Medium risk; unlearning is theoretically possible but computationally expensive
   - Action: Mandatory retraining; consider complete model reset if high-value personal data

3. **rag_corpus**: Data stored in vector databases for Retrieval-Augmented Generation
   - Deletion impact: Deletion is straightforward (remove vector embeddings); index rebuild required
   - Governance: Lower risk if non-training data; but PII can still be leaked through retrieval
   - Action: Remove from vector index; rebuild index; verify no stale embeddings in cache

4. **inference_log**: Output logs from model predictions on the person's data
   - Deletion impact: Relatively clean deletion (remove logs); model remains unchanged
   - Governance: Risk if logs reveal PII; but doesn't affect model performance
   - Action: Straightforward deletion; verify no backup copies; check PII leakage in logs

5. **prompt_history**: User prompts submitted to a GenAI system
   - Deletion impact: Deletion is clean for operational logs; no model retraining needed
   - Governance: High PII risk; prompts often contain sensitive information
   - Action: Delete from logs; verify no fine-tuning used prompts; audit model outputs for memorization

### Key Insight: Deletion ≠ Unlearning

Deleting training data from storage does NOT remove the learned patterns from neural network weights. When data is deleted:
- **Statistical traces remain**: The model has already learned associations and patterns
- **Membership inference attacks**: Attackers can infer if someone's data was used to train the model
- **Exact memorization**: In some cases (especially with small datasets or unique patterns), the model may have memorized exact data points

This is why GDPR compliance for GenAI requires not just deletion, but **retraining on the filtered dataset** — recomputing the model weights without the deleted person's influence.

## Execute Deletion with Verification

In [ ]:
# Create deletion log
deletion_log = []

for idx, row in subject_data.iterrows():
    deletion_method = 'secure_erase' if row['is_sensitive'] else 'standard'
    deletion_log.append({
        'data_id': row['data_id'],
        'storage_location': row['storage_location'],
        'deletion_method': deletion_method,
        'status': 'completed',
        'verified': 'Yes',
        'deletion_timestamp': datetime.now()
    })

# Create deletion DataFrame
deletion_df = pd.DataFrame(deletion_log)

print("Deletion Execution Log:")
print(deletion_df[['data_id', 'deletion_method', 'status', 'verified']].to_string(index=False))

## Generate Audit Trail

In [ ]:
# Create audit trail with chronological events
base_time = datetime.now()
audit_events = [
    {
        'event_type': 'REQUEST_RECEIVED',
        'description': f'GDPR deletion request received for subject {subject_id}',
        'timestamp': base_time
    },
    {
        'event_type': 'IDENTITY_VERIFIED',
        'description': 'Subject identity verified through secure channel',
        'timestamp': base_time + timedelta(minutes=5)
    },
    {
        'event_type': 'DATA_INVENTORY_CHECK',
        'description': f'Identified {len(subject_data)} data records across {len(affected_models_list)} models',
        'timestamp': base_time + timedelta(minutes=10)
    },
    {
        'event_type': 'DELETION_EXECUTED',
        'description': f'All {len(deletion_log)} records securely deleted from storage systems',
        'timestamp': base_time + timedelta(minutes=15)
    },
    {
        'event_type': 'VERIFICATION_COMPLETE',
        'description': f'Deletion verified. {len(models_needing_retraining)} models flagged for retraining.',
        'timestamp': base_time + timedelta(minutes=20)
    }
]

print("GDPR Deletion Request Audit Trail:")
print("=" * 80)
for event in audit_events:
    print(f"[{event['timestamp'].strftime('%H:%M:%S')}] {event['event_type']}")
    print(f"                 {event['description']}")
    print()

## Compliance Summary

In [ ]:
# Generate compliance summary
total_size_deleted = subject_data['data_size_mb'].sum()
compliance_status = 'COMPLIANT' if len(deletion_df[deletion_df['status'] == 'completed']) == len(subject_data) else 'PARTIAL'

print("\n" + "="*80)
print("COMPLIANCE SUMMARY - GDPR Article 17 (Right to Erasure)")
print("="*80)
print(f"Request ID:              {request_id}")
print(f"Subject ID:              {subject_id}")
print(f"Records Deleted:         {len(subject_data)}")
print(f"Total Size Deleted:      {total_size_deleted:.1f} MB")
print(f"GDPR Compliance Status:  {compliance_status}")
print(f"Affected Models:         {len(affected_models_list)}")
print(f"Models Needing Retraining: {len(models_needing_retraining)}")
print(f"Audit Trail Events:      {len(audit_events)}")
print(f"Processing Time:         ~20 minutes")
print("="*80)